# Alzheimer's Severity Prediction Pipeline
### Random Forest → Score Prediction → Ordinal Classification → Diagnosis Lookup

**Pipeline flow:**
1. Train 4 Random Forest models on your CSVs (CDR, MMSE, MoCA, FAQ)
2. User enters their biomarker values
3. RF models predict CDRSB, MMSCORE, MOCA, FAQTOTAL scores
4. Ordinal logistic model classifies severity from those predicted scores
5. Lookup against your diagnosis reference dataset for final insight

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import mean_absolute_error, r2_score, classification_report
from sklearn.preprocessing import OrdinalEncoder

print('Imports done ✓')


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/conda/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/opt/conda/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/opt/conda/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 739, in start
    self.io_loop.start()
  File "/opt/conda/lib/python3.11/site-packages/tornado

AttributeError: _ARRAY_API not found

Imports done ✓


## 1. Configuration — Edit paths here

In [4]:
# ── EDIT THESE PATHS ──────────────────────────────────────────────────────────
DATASETS = [
     {"path": "~/Group5/Data/cdr_reg.csv",  "target": "CDRSB"},
    {"path": "~/Group5/Data/mmse_reg.csv", "target": "MMSCORE"},
    {"path": "~/Group5/Data/moca_reg.csv", "target": "MOCA"},
    {"path": "~/Group5/Data/faq_reg.csv",  "target": "FAQTOTAL"},
]

# Path to your diagnosis reference dataset
# Expected columns: RID, DATE, CDRSB, MMSCORE, MOCA, FAQTOTAL, DIAGNOSIS
DIAGNOSIS_PATH = "~/Group5/Data/dx_reg.csv"
# ─────────────────────────────────────────────────────────────────────────────

FEATURES = {
    "CDRSB": [
        "janssenptau217", "nfl", "entry_age", "abeta42", "APOE4_COUNT"
    ],
    "MMSCORE": [
        "janssenptau217", "nfl", "gfap", "entry_age", "abeta42", "APOE4_COUNT"
    ],
    "MOCA": [
        "janssenptau217", "nfl", "entry_age", "abeta42", "APOE4_COUNT"
    ],
    "FAQTOTAL": [
        "entry_age", "APOE4_COUNT"
    ]
}

# All unique biomarkers needed across all models
ALL_FEATURES = sorted(set(f for feats in FEATURES.values() for f in feats))

print('Configuration set ✓')
print(f'All input features needed from user: {ALL_FEATURES}')

Configuration set ✓
All input features needed from user: ['APOE4_COUNT', 'abeta42', 'entry_age', 'gfap', 'janssenptau217', 'nfl']


## 2. Train Random Forest Models

In [5]:
trained_models = {}   # stores fitted RF per target
feature_medians = {}  # stores training medians for imputing user input

def run_random_forest(path, target):
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()

    print("\n" + "=" * 60)
    print(f"Random Forest → Predicting {target}")

    features = FEATURES[target]
    missing = [f for f in features if f not in df.columns]
    if missing:
        print(f"  Missing columns: {missing}")
        return

    X = df[features].fillna(df[features].median())
    y = df[target]

    # Store medians so we can fill any missing user inputs later
    feature_medians[target] = df[features].median().to_dict()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = RandomForestRegressor(
        n_estimators=300, random_state=42, max_depth=None
    )
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    r2  = r2_score(y_test, preds)

    print(f"  Train: {len(X_train)}  Test: {len(X_test)}")
    print(f"  MAE = {mae:.3f}   R2 = {r2:.3f}")

    print("  Feature importance:")
    imp = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
    for feat, val in imp.items():
        bar = '█' * int(val * 30)
        print(f"    {feat:22} {val:.4f} {bar}")

    # 5-fold CV
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_r2  = cross_val_score(model, X, y, cv=kf, scoring='r2')
    cv_mae = cross_val_score(model, X, y, cv=kf, scoring='neg_mean_absolute_error')
    print(f"  CV R2:  {cv_r2.mean():.3f} +/- {cv_r2.std():.3f}")
    print(f"  CV MAE: {(-cv_mae.mean()):.3f} +/- {cv_mae.std():.3f}")

    trained_models[target] = model


for d in DATASETS:
    run_random_forest(d["path"], d["target"])

print("\nAll models trained ✓")


Random Forest → Predicting CDRSB
  Train: 725  Test: 182
  MAE = 1.101   R2 = 0.198
  Feature importance:
    janssenptau217         0.3857 ███████████
    entry_age              0.2041 ██████
    abeta42                0.1946 █████
    nfl                    0.1871 █████
    APOE4_COUNT            0.0285 
  CV R2:  0.209 +/- 0.078
  CV MAE: 0.988 +/- 0.075


FileNotFoundError: [Errno 2] No such file or directory: '/home/odyssey-comp-05/Group5/Data/reg.csv'

## 3. Train Ordinal / Logistic Severity Classifier
Uses the diagnosis reference dataset to learn severity categories from the 4 scores.

In [ ]:
diag_df = pd.read_csv(DIAGNOSIS_PATH)
diag_df.columns = diag_df.columns.str.strip()

print(f"Diagnosis dataset: {diag_df.shape[0]} rows")
print(f"Columns: {diag_df.columns.tolist()}")
print(f"\nDIAGNOSIS value counts:")
print(diag_df['DIAGNOSIS'].value_counts())

In [ ]:
# Severity score columns used as features for the classifier
SCORE_COLS = ['CDRSB', 'MMSCORE', 'MOCA', 'FAQTOTAL']

clf_df = diag_df[SCORE_COLS + ['DIAGNOSIS']].dropna()

X_clf = clf_df[SCORE_COLS]
y_clf = clf_df['DIAGNOSIS']

X_clf_train, X_clf_test, y_clf_train, y_clf_test = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

# Ordinal logistic regression (multinomial handles ordered classes well)
clf = LogisticRegression(
    multi_class='multinomial',
    solver='lbfgs',
    max_iter=1000,
    random_state=42
)
clf.fit(X_clf_train, y_clf_train)

y_clf_pred = clf.predict(X_clf_test)
print("Classifier performance on test set:")
print(classification_report(y_clf_test, y_clf_pred))

# Store class labels for display
SEVERITY_CLASSES = clf.classes_
print(f"Severity classes: {SEVERITY_CLASSES}")

## 4. User Input & Prediction
Enter biomarker values below — leave any unknown field as `None` and it will be filled with the training median.

In [ ]:
# ── ENTER PATIENT VALUES HERE ─────────────────────────────────────────────────
user_input = {
    "janssenptau217": None,   # e.g. 1.8
    "nfl":            None,   # e.g. 28.5
    "gfap":           None,   # e.g. 210.0
    "entry_age":      None,   # e.g. 74
    "abeta42":        None,   # e.g. 9.2
    "APOE4_COUNT":    None,   # 0, 1, or 2
}
# ─────────────────────────────────────────────────────────────────────────────

print("User input received:")
for k, v in user_input.items():
    status = str(v) if v is not None else '(will use training median)'
    print(f"  {k:22} {status}")

In [ ]:
# Step 1: Predict all 4 scores using the trained RF models
predicted_scores = {}

print("\nPredicted clinical scores from biomarkers:")
print("-" * 45)

for target, model in trained_models.items():
    features = FEATURES[target]
    medians  = feature_medians[target]

    # Build input row — fill None with training median
    row = {}
    for f in features:
        val = user_input.get(f)
        row[f] = val if val is not None else medians[f]

    X_user = pd.DataFrame([row])[features]
    pred   = model.predict(X_user)[0]
    predicted_scores[target] = round(float(pred), 2)

    print(f"  {target:12}  →  {predicted_scores[target]:.2f}")

print("-" * 45)

In [ ]:
# Step 2: Pass predicted scores into the ordinal classifier
score_row = pd.DataFrame([[
    predicted_scores.get('CDRSB',    0),
    predicted_scores.get('MMSCORE',  0),
    predicted_scores.get('MOCA',     0),
    predicted_scores.get('FAQTOTAL', 0),
]], columns=SCORE_COLS)

severity_pred  = clf.predict(score_row)[0]
severity_proba = clf.predict_proba(score_row)[0]

print("\nSeverity classification:")
print(f"  Predicted severity:  {severity_pred}")
print("\n  Probability per class:")
for cls, prob in zip(SEVERITY_CLASSES, severity_proba):
    bar = '█' * int(prob * 30)
    print(f"    {str(cls):20}  {prob:.3f}  {bar}")

In [ ]:
# Step 3: Look up similar patients in the diagnosis reference dataset
# Find the 5 closest patients by Euclidean distance on the 4 score columns

ref = diag_df[SCORE_COLS + ['DIAGNOSIS', 'RID', 'DATE']].dropna().copy()

for col in SCORE_COLS:
    ref[col] = pd.to_numeric(ref[col], errors='coerce')
ref = ref.dropna(subset=SCORE_COLS)

# Euclidean distance from user's predicted scores to every row
ref['distance'] = np.sqrt(
    (ref['CDRSB']    - predicted_scores.get('CDRSB',    0)) ** 2 +
    (ref['MMSCORE']  - predicted_scores.get('MMSCORE',  0)) ** 2 +
    (ref['MOCA']     - predicted_scores.get('MOCA',     0)) ** 2 +
    (ref['FAQTOTAL'] - predicted_scores.get('FAQTOTAL', 0)) ** 2
)

top5 = ref.nsmallest(5, 'distance')[['RID', 'DATE', 'CDRSB', 'MMSCORE', 'MOCA', 'FAQTOTAL', 'DIAGNOSIS', 'distance']]

print("\n5 most similar patients in reference dataset:")
display(top5.reset_index(drop=True))

## 5. Final Summary

In [ ]:
print("\n" + "=" * 60)
print("  PATIENT SUMMARY")
print("=" * 60)

print("\nBiomarker inputs used:")
for k, v in user_input.items():
    display_val = v if v is not None else f"(median fallback)"
    print(f"  {k:22} {display_val}")

print("\nPredicted clinical scores:")
score_labels = {
    'CDRSB':    ('CDR Sum of Boxes',       '0-18',  'higher = worse'),
    'MMSCORE':  ('Mini-Mental State Exam',  '0-30',  'lower = worse'),
    'MOCA':     ('Montreal Cog Assessment', '0-30',  'lower = worse'),
    'FAQTOTAL': ('Functional Activities Q', '0-30',  'higher = worse'),
}
for col, (label, scale, direction) in score_labels.items():
    val = predicted_scores.get(col, 'N/A')
    print(f"  {label:30} {val:6.2f}   (scale {scale}, {direction})")

print(f"\nSeverity classification:  {severity_pred}")
top_class_prob = max(severity_proba)
print(f"Confidence:               {top_class_prob:.1%}")

# Interpretation guide
print("\nInterpretation guide:")
interp = {
    'NL':        'No cognitive impairment detected.',
    'MCI':       'Mild Cognitive Impairment — monitor closely, not yet dementia.',
    'Dementia':  'Dementia-level impairment — clinical follow-up strongly advised.',
}
for cls, desc in interp.items():
    marker = '>>>' if cls == str(severity_pred) else '   '
    print(f"  {marker} {cls:12}  {desc}")

print("\nNote: This tool is for research purposes only.")
print("      It does not constitute a clinical diagnosis.")
print("=" * 60)